# c07：从 Elasticsearch 拉取 chunk 文档

按 **文件名**（`source_file`）、**路径子串**（`source_path`）、**chunk_version** 等条件筛选，拉取命中文档的 **全部 `_source` 字段**，并做可读展示。

**前置**：项目根 `.env` 中 `ES_*` 可达；`uv sync`（含 `elasticsearch` 客户端）。在 `notebooks/` 下运行本笔记本时，下方首格会自动把仓库根加入 `sys.path`。

**字段说明**（与 [mapping](../src/es_store/mapping.py) 一致）：`text`、`embedding`、`source_file`、`source_path`、`chunk_index`、`char_start`/`char_end`、`chunk_type`、`mime_type`、`doc_type`、`domain`、`source_doc_id`、`source_sha256`、`source_oss_url`、`chunk_version`、`page_start`/`page_end`、`extra`。


In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)
print("仓库根目录:", _ROOT)


仓库根目录: /Users/zheng/projects/python/ai/rag/rag_law


## 1. 筛选条件（按需修改）

- `FILTER_SOURCE_FILE`：精确匹配 `source_file`（keyword），例如 `md3__民法典.md`。
- `FILTER_SOURCE_PATH_CONTAINS`：路径包含某子串时用 wildcard（如 `民法典` → `*民法典*`）。
- `FILTER_CHUNK_VERSION`：精确匹配版本标签；`None` 表示不按版本过滤。
- `REQUIRE_FILTERS`：为 `True` 时至少填一项筛选；为 `False` 时可 `match_all`（慎用，请配合较小的 `MAX_HITS`）。


In [7]:
# --- 按需修改 ---
FILTER_SOURCE_FILE: str | None = "md3__婚姻法.md"
FILTER_SOURCE_PATH_CONTAINS: str | None = None
FILTER_CHUNK_VERSION: str | None = None

MAX_HITS = 200
REQUIRE_FILTERS = True

# 展示：正文预览长度；向量只显示维度不打印数组
TEXT_PREVIEW_CHARS = 2000
SHOW_EMBEDDING_STATS = True  # True：显示 min/max/mean；False：仅维度


## 2. 连接 ES 并查询

In [8]:
from conf.settings import get_settings
from es_store.client import elasticsearch_client

get_settings.cache_clear()
settings = get_settings()
client = elasticsearch_client(settings)
index = settings.es_index

assert client.ping(), "ES 不可达，请检查服务与 .env 中 ES_HOST/ES_PORT"

must: list = []
if FILTER_SOURCE_FILE:
    must.append({"term": {"source_file": FILTER_SOURCE_FILE}})
if FILTER_SOURCE_PATH_CONTAINS:
    pat = f"*{FILTER_SOURCE_PATH_CONTAINS}*"
    must.append({"wildcard": {"source_path": pat}})
if FILTER_CHUNK_VERSION:
    must.append({"term": {"chunk_version": FILTER_CHUNK_VERSION}})

if REQUIRE_FILTERS and not must:
    raise ValueError("未设置任何筛选条件。请设置 FILTER_* 或将 REQUIRE_FILTERS 设为 False")

query = {"bool": {"must": must}} if must else {"match_all": {}}

resp = client.search(
    index=index,
    query=query,
    size=MAX_HITS,
    sort=[{"source_path": "asc"}, {"chunk_index": "asc"}],
)
hits = resp.get("hits", {}).get("hits", [])
total = resp.get("hits", {}).get("total", {})
if isinstance(total, dict):
    total_val = total.get("value", len(hits))
else:
    total_val = total

print(f"索引: {index}")
print(f"命中条数（本页）: {len(hits)}  total_relation: {total_val}")


索引: rag_law_chunks_c06
命中条数（本页）: 4  total_relation: 4


## 3. 人性化展示

每条文档单独一块：`embedding` 不刷屏，只显示维度与可选统计量；`text` 可截断预览。


In [9]:
import html

from IPython.display import HTML, display


def _fmt_embedding(vec: list) -> str:
    if not isinstance(vec, list) or not vec:
        return repr(vec)
    n = len(vec)
    if not SHOW_EMBEDDING_STATS:
        return f"[dense_vector len={n}]"
    xs = [float(x) for x in vec]
    return f"[dense_vector len={n} min={min(xs):.6f} max={max(xs):.6f} mean={sum(xs)/n:.6f}]"


def _fmt_value(key: str, val) -> str:
    if key == "embedding":
        return _fmt_embedding(val)  # type: ignore[arg-type]
    if key == "text" and isinstance(val, str) and len(val) > TEXT_PREVIEW_CHARS:
        head = val[:TEXT_PREVIEW_CHARS]
        return f"{head}\n…（共 {len(val)} 字，已截断预览；调大 TEXT_PREVIEW_CHARS 可看更多）"
    if isinstance(val, (dict, list)):
        import json

        try:
            return json.dumps(val, ensure_ascii=False, indent=2)
        except TypeError:
            return repr(val)
    return repr(val)


parts: list[str] = []
parts.append(f"<h2>共 {len(hits)} 条</h2>")

for i, hit in enumerate(hits):
    hid = hit.get("_id", "")
    score = hit.get("_score")
    src = hit.get("_source") or {}
    block = [
        "<div style='border:1px solid #334155;border-radius:8px;padding:12px;margin-bottom:16px;background:#0f172a;color:#e2e8f0;font-family:ui-monospace,monospace;font-size:13px;'>"
    ]
    block.append(
        "<div style='color:#94a3b8;margin-bottom:8px;'>#%s &nbsp; <code>_id</code>=%s"
        % (i + 1, html.escape(repr(hid)))
    )
    if score is not None:
        block.append(" &nbsp; <code>_score</code>=%s" % html.escape(repr(score)))
    block.append("</div>")
    for key in sorted(src.keys()):
        val = src[key]
        body = html.escape(_fmt_value(key, val))
        block.append(
            "<p style='margin:6px 0;'><span style='color:#38bdf8;'>%s</span>:</p>"
            % html.escape(key)
        )
        block.append(
            "<pre style='white-space:pre-wrap;word-break:break-word;margin:0 0 8px 12px;'>%s</pre>"
            % body
        )
    block.append("</div>")
    parts.append("".join(block))

display(HTML("".join(parts)))


## 4.（可选）导出为 JSON

便于在外部 diff 或存档。


In [ ]:
import json
from pathlib import Path

out_path = _ROOT / "notebooks" / "_c07_es_export.json"
payload = [{"_id": h.get("_id"), "_score": h.get("_score"), "_source": h.get("_source")} for h in hits]
# 导出时省略 embedding 数组体积；需要全量向量可删除下方替换逻辑
slim = []
for row in payload:
    src = row.get("_source") or {}
    s = dict(src)
    emb = s.get("embedding")
    if isinstance(emb, list):
        s["embedding"] = "[omitted dense_vector len=%s]" % len(emb)
    slim.append({"_id": row["_id"], "_score": row["_score"], "_source": s})

out_path.write_text(json.dumps(slim, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写入:", out_path)
